# Guidance extraction on Colab

Runs `nowcast guidance` with an open model served by vLLM on the Colab GPU.

Before starting: **Runtime → Change runtime type → GPU** (L4 or A100 if offered; T4 also works for the 7B model).
Upload `altdata-data.zip` to a Google Drive folder named `altdata`.

Extractions are cached in Drive, so if the session disconnects, rerun all cells and it continues.

In [ ]:
MODEL = "Qwen/Qwen2.5-7B-Instruct"
RUN = "--limit 20"   # then "--sample data/labels/labels.csv", then "" for all releases
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/altdata"
!ls -lh {DRIVE}

In [ ]:
%cd /content
![ -d altdata-nowcast ] || git clone -q https://github.com/nirzareee/altdata-nowcast.git
%cd /content/altdata-nowcast
!git pull -q
![ -d data/raw/releases ] || unzip -q -o {DRIVE}/altdata-data.zip
!mkdir -p {DRIVE}/guidance data/raw && rm -rf data/raw/guidance && ln -s {DRIVE}/guidance data/raw/guidance
!ls data/raw/releases | wc -l

In [ ]:
!pip install -q -e . vllm

In [ ]:
import subprocess, time, urllib.request
log = open("/content/vllm.log", "w")
server = subprocess.Popen(
    ["vllm", "serve", MODEL, "--dtype", "half", "--max-model-len", "8192",
     "--gpu-memory-utilization", "0.9", "--port", "8000"],
    stdout=log, stderr=subprocess.STDOUT)
for _ in range(120):
    try:
        urllib.request.urlopen("http://localhost:8000/health"); print("server ready"); break
    except Exception:
        if server.poll() is not None:
            raise RuntimeError("vLLM exited; see /content/vllm.log")
        time.sleep(10)
else:
    raise TimeoutError("server did not start in 20 minutes; see /content/vllm.log")

In [ ]:
!nowcast guidance --base-url http://localhost:8000/v1 --model {MODEL} --workers 16 {RUN}

In [ ]:
!cp data/processed/guidance_*.parquet {DRIVE}/
!ls -lh {DRIVE}/*.parquet

In [ ]:
# If anything failed:
!tail -30 /content/vllm.log